# Using CytoTable or CytoDataframe data in Pycytominer

This walkthrough shows two ways to bring parquet-backed profiling data into pycytominer and run the same downstream normalization workflow.

The first example starts from a standalone CytoDataframe parquet file in `tests/test_data/cytodataframe`. The second starts from a CytoTable-style warehouse in `tests/test_data/cytotable`, where `load_profiles()` automatically resolves the profile table from the warehouse layout.

Both examples also show how OME-Arrow image payload columns can live alongside profile data. Pycytominer does not normalize those nested payload columns as features. Instead, inferred normalization keeps the numeric profile measurements, and normalized outputs preserve the image payload columns unchanged.

In [1]:
from pathlib import Path

import pyarrow.parquet as pq
import pyarrow.types as patypes

from pycytominer import normalize
from pycytominer.cyto_utils import load_profiles

# Point to the example inputs used in the test suite.
# Notebook execution starts from the repository root, so these are repo-relative paths.
single_file_path = Path("../tests/test_data/cytodataframe/example.ome.parquet")
warehouse_path = Path("../tests/test_data/cytotable/examplehuman_iceberg_warehouse")

In [2]:
# The standalone parquet file includes OME-Arrow payload columns stored as Arrow structs.
image_schema = pq.read_table(single_file_path).schema
struct_columns = [field.name for field in image_schema if patypes.is_struct(field.type)]
struct_columns

['Image_FileName_GFP_OMEArrow_ORIG',
 'Image_FileName_GFP_OMEArrow_COMP',
 'Image_FileName_RFP_OMEArrow_ORIG',
 'Image_FileName_RFP_OMEArrow_LABL',
 'Image_FileName_RFP_OMEArrow_COMP',
 'Image_FileName_DAPI_OMEArrow_ORIG',
 'Image_FileName_DAPI_OMEArrow_LABL',
 'Image_FileName_DAPI_OMEArrow_COMP']

In [3]:
# Start from the standalone parquet file.
single_file_df = load_profiles(single_file_path)

# Create a simple treatment label so normalization has a control group.
# Use row position instead of image number because these small examples do not
# necessarily contain multiple image-number values.
single_file_df = single_file_df.assign(
    Metadata_treatment=[
        "control" if row_number % 2 == 0 else "drug"
        for row_number in range(len(single_file_df))
    ]
)

# This file is mostly an OME-Arrow example, not a full CellProfiler profile table.
# Use explicit feature selection here because `infer` correctly skips the OME-Arrow
# payload columns, but there are not enough CellProfiler-style feature columns left.
single_file_normalized_df = normalize(
    profiles=single_file_df,
    features=["Metadata_Cells_Number_Object_Number"],
    meta_features=["Metadata_ImageNumber", "Metadata_treatment"],
    samples="Metadata_treatment == 'control'",
    method="standardize",
)

single_file_normalized_df

,Metadata_ImageNumber,Metadata_treatment,Image_FileName_GFP,Image_FileName_RFP,Image_FileName_DAPI,Image_FileName_GFP_OMEArrow_ORIG,Image_FileName_GFP_OMEArrow_LABL,Image_FileName_GFP_OMEArrow_COMP,Image_FileName_RFP_OMEArrow_ORIG,Image_FileName_RFP_OMEArrow_LABL,Image_FileName_RFP_OMEArrow_COMP,Image_FileName_DAPI_OMEArrow_ORIG,Image_FileName_DAPI_OMEArrow_LABL,Image_FileName_DAPI_OMEArrow_COMP,Metadata_Cells_Number_Object_Number
353,31,control,B7_01_2_3_GFP_001.tif,B7_01_3_3_RFP_001.tif,B7_01_1_3_DAPI_001.tif,{'acquisition_datetime': 2026-03-30 17:41:02.3...,None,{'acquisition_datetime': 2026-03-30 17:41:02.3...,{'acquisition_datetime': 2026-03-30 17:41:02.6...,{'acquisition_datetime': 2026-03-30 17:41:02.6...,{'acquisition_datetime': 2026-03-30 17:41:02.6...,{'acquisition_datetime': 2026-03-30 17:41:03.0...,{'acquisition_datetime': 2026-03-30 17:41:03.0...,{'acquisition_datetime': 2026-03-30 17:41:03.0...,-1.0
1564,113,drug,H12_01_2_1_GFP_001.tif,H12_01_3_1_RFP_001.tif,H12_01_1_1_DAPI_001.tif,{'acquisition_datetime': 2026-03-30 17:41:02.4...,None,{'acquisition_datetime': 2026-03-30 17:41:02.4...,{'acquisition_datetime': 2026-03-30 17:41:02.7...,{'acquisition_datetime': 2026-03-30 17:41:02.7...,{'acquisition_datetime': 2026-03-30 17:41:02.7...,{'acquisition_datetime': 2026-03-30 17:41:03.1...,{'acquisition_datetime': 2026-03-30 17:41:03.1...,{'acquisition_datetime': 2026-03-30 17:41:03.1...,25.0
1275,94,control,F7_01_2_2_GFP_001.tif,F7_01_3_2_RFP_001.tif,F7_01_1_2_DAPI_001.tif,{'acquisition_datetime': 2026-03-30 17:41:02.5...,None,{'acquisition_datetime': 2026-03-30 17:41:02.5...,{'acquisition_datetime': 2026-03-30 17:41:02.9...,{'acquisition_datetime': 2026-03-30 17:41:02.9...,{'acquisition_datetime': 2026-03-30 17:41:02.9...,{'acquisition_datetime': 2026-03-30 17:41:03.2...,{'acquisition_datetime': 2026-03-30 17:41:03.2...,{'acquisition_datetime': 2026-03-30 17:41:03.2...,1.0


In [4]:
# The warehouse image data include OME-Arrow payload columns stored as Arrow structs.
warehouse_image_path = warehouse_path / "warehouse" / "images" / "source_images"
image_schema = pq.read_table(warehouse_image_path).schema
struct_columns = [field.name for field in image_schema if patypes.is_struct(field.type)]
struct_columns

['ome_arrow_image', 'ome_arrow_outline', 'ome_arrow_mask', 'ome_arrow_label']

The CytoTable-style warehouse profile table is the place to use `features="infer"`. It contains standard CellProfiler-style feature columns, while the OME-Arrow payloads live separately in the warehouse image data.

In [5]:
# Start from the warehouse root.
# `load_profiles()` detects the CytoTable-style profiles table automatically.
warehouse_profiles_df = load_profiles(warehouse_path)

# Create the same toy treatment label used in the single-file example.
# Use row position instead of image number because this warehouse sample only
# contains one image-number value.
warehouse_profiles_df = warehouse_profiles_df.assign(
    Metadata_treatment=[
        "control" if row_number % 2 == 0 else "drug"
        for row_number in range(len(warehouse_profiles_df))
    ]
)

# The warehouse profile table can use the same normalize call.
# Here `infer` keeps the numeric CellProfiler-style features and ignores the
# OME-Arrow payload content shown above.
warehouse_normalized_df = normalize(
    profiles=warehouse_profiles_df,
    features="infer",
    meta_features="infer",
    samples="Metadata_treatment == 'control'",
    method="standardize",
)

warehouse_normalized_df.shape

(289, 314)

`normalize()` returns only the inferred metadata and profile feature columns, so nested OME-Arrow image payload columns are left out of the normalized output while the numeric morphology data are processed as usual.

In [6]:
# Show a representative subset of metadata and morphology columns without regex filtering.
display_columns = [
    column
    for column in warehouse_normalized_df.columns
    if column.startswith("Cells_")
    or column.startswith("Cytoplasm_")
    or column.startswith("Nuclei_")
]

warehouse_normalized_df[display_columns[:12]].head()

,Cytoplasm_AreaShape_Area,Cytoplasm_AreaShape_BoundingBoxArea,Cytoplasm_AreaShape_Center_X,Cytoplasm_AreaShape_Center_Y,Cytoplasm_AreaShape_Compactness,Cytoplasm_AreaShape_Eccentricity,Cytoplasm_AreaShape_EquivalentDiameter,Cytoplasm_AreaShape_EulerNumber,Cytoplasm_AreaShape_Extent,Cytoplasm_AreaShape_FormFactor,Cytoplasm_AreaShape_MajorAxisLength,Cytoplasm_AreaShape_MaxFeretDiameter
0,-0.245630,-0.450081,1.266428,-2.067002,-0.503010,-0.078521,-0.130853,0.0,0.641023,0.499198,-0.416841,-0.164777
1,-0.401800,-0.712920,1.452798,-2.057679,0.737880,-0.319813,-0.318141,0.0,1.139864,-1.008226,-0.435693,-0.289696
2,-1.397379,-1.413825,1.065620,-2.042053,0.127083,-1.079252,-2.014114,0.0,-2.063880,-0.377761,-1.536861,-1.830915
3,0.623062,0.531186,-1.355541,-2.056849,0.277391,-0.140494,0.759976,0.0,0.439491,-0.549362,0.355059,0.606930
4,-0.206588,-0.077100,-1.506388,-2.008949,-0.012417,0.532796,-0.085682,0.0,-0.424019,-0.207110,-0.015245,-0.127986
